# 棋力 CNN 訓練

執行 `aicup.train_rank` 訓練 `GoRankResNet`，checkpoint 存到 Drive 的 `rank-trained-models/<名稱>/`，結束後畫出每個 epoch 的數值。

- 全量特徵要 30 GB 以上系統 RAM
- 依序執行全部 cell；中斷後重跑同一個 MODE 會從 `last.pt` 接續，接續時 `COMMIT` 要和開始時相同

| MODE | 內容 | 競賽一最終版本 |
|---|---|---|
| `smoke` | 10% 特徵、200 步，確認環境 | |
| `rank_cnn` | 教學版切分（val 100,000 局）、D4 增強、cosine、15 epoch | A 組段位相容度用的 CNN |
| `rank_cnn_v180000` | 同上，val 擴大到 180,000 局，`--patience 0`、seed 42 | Model B 的主幹初始化 |
| `e2e_smoke`、`e2e_full` | `aicup.train_player` 端對端玩家模型 | 不在最終版本流程內 |

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `rank-features-10pct/` | `smoke` | `python -m aicup.extract_features --train-dir training-v1.1/training --frac 0.1 --out rank-features-10pct` |
| `rank-features/` | `rank_cnn` | `python -m aicup.extract_features --train-dir training-v1.1/training` |
| `rank-features-v180000/` | `rank_cnn_v180000`、`e2e_*` | `python -m aicup.extract_features --train-dir training-v1.1/training --val-size 180000 --out rank-features-v180000` |
| `rank-trained-models/rank-cnn-v180000/best_model.pth` | `e2e_*` | 這份 notebook 的 `rank_cnn_v180000` |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'smoke'
COMPARE = []  # 一起畫圖比較的 checkpoint 目錄名，例如 ['rank-cnn']
AUTO_UNASSIGN = True
E2E_INIT = 'rank-trained-models/rank-cnn-v180000/best_model.pth'

RUNS = {  # MODE: (特徵目錄, checkpoint 目錄, 模組, 參數)
    'smoke': ('rank-features-10pct', 'rank-trained-models/smoke', 'aicup.train_rank',
              ['--epochs', '1', '--max-steps', '200', '--fresh']),
    'rank_cnn': ('rank-features', 'rank-trained-models/rank-cnn', 'aicup.train_rank',
                 ['--augment', '--schedule', 'cosine', '--epochs', '15']),
    'rank_cnn_v180000': ('rank-features-v180000', 'rank-trained-models/rank-cnn-v180000', 'aicup.train_rank',
                         ['--augment', '--schedule', 'cosine', '--epochs', '15', '--patience', '0', '--seed', '42']),
    'e2e_smoke': ('rank-features-v180000', 'rank-trained-models/smoke-e2e', 'aicup.train_player',
                  ['--init', E2E_INIT, '--epochs', '1', '--max-steps', '200', '--fresh', '--accum', '4',
                   '--warmup-steps', '50', '--monitor-every', '25', '--log-every', '25', '--augment']),
    'e2e_full': ('rank-features-v180000', 'rank-trained-models/e2e', 'aicup.train_player',
                 ['--init', E2E_INIT, '--epochs', '5', '--lr', '1e-4', '--warmup-steps', '50', '--accum', '4',
                  '--schedule', 'cosine', '--augment', '--seed', '42']),
}
HINTS = {
    'rank-features-10pct': 'python -m aicup.extract_features --train-dir training-v1.1/training --frac 0.1 --out rank-features-10pct',
    'rank-features': 'python -m aicup.extract_features --train-dir training-v1.1/training',
    'rank-features-v180000': ('python -m aicup.extract_features --train-dir training-v1.1/training '
                              '--val-size 180000 --out rank-features-v180000'),
}
if MODE not in RUNS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(RUNS))
FEAT_DIR, CKPT_DIR, MODULE, ARGS = RUNS[MODE]
CMD = ['python', '-m', MODULE, '--feat-dir', FEAT_DIR, '--ckpt-dir', CKPT_DIR, *ARGS]
NEEDS = [(FEAT_DIR + '/train', HINTS[FEAT_DIR]), (FEAT_DIR + '/val', HINTS[FEAT_DIR])]
if MODE.startswith('e2e'):
    NEEDS.append((E2E_INIT, '這份 notebook 的 MODE rank_cnn_v180000'))
MIN_RAM_GB = 4 if MODE == 'smoke' else 30  # 載入打包特徵所需的系統 RAM
print(' '.join(CMD))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：特徵目錄的連結、需要的檔案都在、GPU 與 VRAM 檢查、系統 RAM 足夠。checkpoint 目錄已用別的 commit 訓練過時停止。

In [ ]:
link_drive([FEAT_DIR, 'rank-trained-models'])
need(NEEDS)
check_commit(CKPT_DIR, GIT_HEAD)
from aicup.colab_setup import check_gpu

check_gpu(FEAT_DIR)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3
print('系統 RAM %.1f GB，%s 需要 %d GB' % (ram_gb, FEAT_DIR, MIN_RAM_GB))
if ram_gb < MIN_RAM_GB:
    raise SystemExit('RAM 不足，載入特徵時 kernel 會被 kill')

## 執行

輸出同時寫到 Drive 的 `<checkpoint 目錄>/train.log`，runtime 釋放後仍能看錯誤訊息。

In [ ]:
run(CMD, os.path.join(CKPT_DIR, 'train.log'))

## 檢查

要看到：每個 epoch 的 loss、accuracy、val 分數與耗時。看舊結果時把 `CKPT_DIR` 改成 `rank-trained-models/<名稱>`。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

runs = {}
for d in [CKPT_DIR, *('rank-trained-models/' + c for c in COMPARE)]:
    path = os.path.join(d, 'history.csv')
    if os.path.exists(path):
        runs[os.path.basename(d)] = pd.read_csv(path)
    else:
        print('找不到', path)

if runs:
    panels = [('Loss', [('train_loss', '-'), ('val_loss', '--')]),
              ('Accuracy (%)', [('train_acc', '-'), ('val_acc', '--')]),
              ('Val score', [('val_score_pick', '-'), ('val_score_argmax', '--')]),
              ('Minutes per epoch', [('total_minutes', '-')])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (title, cols) in zip(axes.flat, panels):
        for i, (name, h) in enumerate(runs.items()):
            for col, style in cols:
                if col in h:
                    ax.plot(h['epoch'], h[col], style, color='C%d' % i, marker='o', label='%s %s' % (name, col))
        ax.set_title(title)
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(CKPT_DIR, 'history.png'), dpi=120)
    plt.show()
    for name, h in runs.items():
        print(name)
        display(h.round(4))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')